In [2]:
import pandas as pd
import subprocess
import os

# Load station list
stations = pd.read_csv('isd-history.csv')
oregon = stations[stations['STATE'] == 'OR'].copy()

# Format the filename the way the bucket uses it
oregon['USAF'] = oregon['USAF'].astype(str).str.zfill(6)
oregon['WBAN'] = oregon['WBAN'].astype(str).str.zfill(5)
oregon['filename'] = oregon['USAF'] + oregon['WBAN'] + '.csv'

print(f"Found {len(oregon)} Oregon stations")
print(oregon[['USAF', 'WBAN', 'STATION NAME', 'filename']].to_string())

Found 113 Oregon stations
         USAF   WBAN                                 STATION NAME         filename
14564  690330  99999                            HOOD CANAL BRIDGE  69033099999.csv
17165  720202  00118                            TILLAMOOK AIRPORT  72020200118.csv
17166  720202  99999                                TILLAMOOK AWS  72020299999.csv
17342  720365  24267                            BROOKINGS AIRPORT  72036524267.csv
17343  720365  99999                            BROOKINGS AIRPORT  72036599999.csv
17593  720638  00224                       BEND MUNICIPAL AIRPORT  72063800224.csv
17691  720753  99999                                KEN JERNSTEDT  72075399999.csv
17709  720837  99999                                 JOSEPH STATE  72083799999.csv
20406  725891  99999                                     LAKEVIEW  72589199999.csv
20407  725895  94236                        KLAMATH FALLS AIRPORT  72589594236.csv
20408  725895  99999                                KLAMATH F

In [4]:
import pandas as pd
import subprocess
import os

# Load station list and filter Oregon
stations = pd.read_csv('isd-history.csv')
oregon = stations[stations['STATE'] == 'OR'].copy()

# Format filenames
oregon['USAF'] = oregon['USAF'].astype(str).str.zfill(6)
oregon['WBAN'] = oregon['WBAN'].astype(str).str.zfill(5)
oregon['filename'] = oregon['USAF'] + oregon['WBAN'] + '.csv'

# Create output folder
os.makedirs('oregon_weather', exist_ok=True)

years = range(1970, 2022)
total = len(years) * len(oregon)
count = 0
skipped = 0

for year in years:
    for _, row in oregon.iterrows():
        filename = row['filename']
        s3_path = f"s3://noaa-gsod-pds/{year}/{filename}"
        local_path = f"oregon_weather/{year}_{filename}"

        # Skip if already downloaded
        if os.path.exists(local_path):
            count += 1
            continue

        result = subprocess.run(
            ["aws", "s3", "cp", "--no-sign-request", s3_path, local_path],
            capture_output=True
        )

        if result.returncode != 0:
            skipped += 1  # Station didn't exist that year, that's normal
        
        count += 1

    print(f"✅ {year} done — {count}/{total} processed, {skipped} missing (normal)")

print(" Download complete! All files saved to oregon_weather/")

✅ 1970 done — 113/5876 processed, 101 missing (normal)
✅ 1971 done — 226/5876 processed, 202 missing (normal)
✅ 1972 done — 339/5876 processed, 304 missing (normal)
✅ 1973 done — 452/5876 processed, 397 missing (normal)
✅ 1974 done — 565/5876 processed, 492 missing (normal)
✅ 1975 done — 678/5876 processed, 582 missing (normal)
✅ 1976 done — 791/5876 processed, 669 missing (normal)
✅ 1977 done — 904/5876 processed, 754 missing (normal)
✅ 1978 done — 1017/5876 processed, 842 missing (normal)
✅ 1979 done — 1130/5876 processed, 930 missing (normal)
✅ 1980 done — 1243/5876 processed, 1016 missing (normal)
✅ 1981 done — 1356/5876 processed, 1105 missing (normal)
✅ 1982 done — 1469/5876 processed, 1193 missing (normal)
✅ 1983 done — 1582/5876 processed, 1281 missing (normal)
✅ 1984 done — 1695/5876 processed, 1367 missing (normal)
✅ 1985 done — 1808/5876 processed, 1454 missing (normal)
✅ 1986 done — 1921/5876 processed, 1542 missing (normal)
✅ 1987 done — 2034/5876 processed, 1629 missing (

In [6]:
import pandas as pd
import glob
import os

# Find all downloaded files
all_files = glob.glob('oregon_weather/*.csv')
print(f"Found {len(all_files)} files to combine...")

dfs = []
skipped = 0

for i, f in enumerate(all_files):
    try:
        df = pd.read_csv(f)
        
        # Extract year and station from filename
        basename = os.path.basename(f)  # e.g. "1970_72020200118.csv"
        parts = basename.replace('.csv', '').split('_')
        df['YEAR_FILE'] = parts[0]
        df['STATION_FILE'] = parts[1]
        
        dfs.append(df)
    except Exception as e:
        skipped += 1

    if (i + 1) % 500 == 0:
        print(f"  Processed {i+1}/{len(all_files)} files...")

print(f"Combining {len(dfs)} files ({skipped} skipped due to errors)...")
combined = pd.concat(dfs, ignore_index=True)

# Convert DATE to datetime
combined['DATE'] = pd.to_datetime(combined['DATE'], errors='coerce')

# Sort by station and date
combined = combined.sort_values(['STATION_FILE', 'DATE']).reset_index(drop=True)

print(f"✅ Combined shape: {combined.shape}")
print(f"   Date range: {combined['DATE'].min()} to {combined['DATE'].max()}")
print(f"   Columns: {list(combined.columns)}")

# Save to CSV
output_path = 'oregon_weather_1970_2021.csv'
combined.to_csv(output_path, index=False)
print(f"🎉 Saved to {output_path}!")

Found 1533 files to combine...
  Processed 500/1533 files...
  Processed 1000/1533 files...
  Processed 1500/1533 files...
Combining 1533 files (0 skipped due to errors)...
✅ Combined shape: (509866, 30)
   Date range: 1970-01-01 00:00:00 to 2021-12-31 00:00:00
   Columns: ['STATION', 'DATE', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'TEMP', 'TEMP_ATTRIBUTES', 'DEWP', 'DEWP_ATTRIBUTES', 'SLP', 'SLP_ATTRIBUTES', 'STP', 'STP_ATTRIBUTES', 'VISIB', 'VISIB_ATTRIBUTES', 'WDSP', 'WDSP_ATTRIBUTES', 'MXSPD', 'GUST', 'MAX', 'MAX_ATTRIBUTES', 'MIN', 'MIN_ATTRIBUTES', 'PRCP', 'PRCP_ATTRIBUTES', 'SNDP', 'FRSHTT', 'YEAR_FILE', 'STATION_FILE']
🎉 Saved to oregon_weather_1970_2021.csv!
